##DOWNLOAD THE VTUAD INTO THE INPUTS FOLDER
##CREATE A CUSTOM ONC DATASET AND ALSO PLACE INTO INPUTS FOLDER

##ONC DATASET SETTINGS

In [ ]:
#......

##2000_4000 processing

In [ ]:
# This script constructs a split within each class of each ship ID to prevent leakage


import csv
import shutil
from collections import defaultdict
from pathlib import Path


ROOT = Path("/Users/mandeepwalia/Downloads/Vessel-Classification-Representations-Architectures-and-Hyperparameters-main/Inputs/......") #MAKE SURE TO HAVE THE DATASET INSTALLED IN THIS FOLDER
OUT = Path("Outputs/2000_4000/2000_4000_splits") #MAKE SURE TO PASTE YOUR OUTPUTS PATH BEFORE 2000_4000_splits
SPLITS = ["train", "validation", "test"]
MOVE = False
SPLIT_ORDER = {"train": 0, "validation": 1, "test": 2}


def mmsi_to_folder(mmsi_raw): #Normalize MMSI to integer for clean folder name
    s = str(mmsi_raw).strip()
    try:
        f = float(s)
        if f.is_integer():
            return str(int(f))
    except ValueError:
        pass
    return s.replace("/", "_")


def main():
    if not ROOT.is_dir():
        raise SystemExit(f"root not found: {ROOT}")

    # Key: Obtain (MMSI, Class)
    # Value: list of tuples: (split order, file index, and source_wav_path)
    groups = defaultdict(list)
    total_rows = 0
    missing = []

    for split in SPLITS:
        csv_path = ROOT / split / f"metadata_{split}.csv"
        audio_dir = ROOT / split / "audio"
        if not csv_path.is_file():
            raise SystemExit(f"metadata not found: {csv_path}")
        with open(csv_path, newline="") as f:
            reader = csv.DictReader(f)
            for row in reader:
                total_rows += 1
                cls = row["label"].strip()
                mmsi = mmsi_to_folder(row["MMSI"])
                fidx = row["file_index"].strip()
                src = audio_dir / cls / f"{fidx}.wav"
                if not src.is_file():
                    missing.append(str(src))
                    continue
                try:
                    order_key = (SPLIT_ORDER[split], int(fidx))
                except ValueError:
                    order_key = (SPLIT_ORDER[split], fidx)
                groups[(cls, mmsi)].append((order_key, src))

    print(f"read {total_rows} rows across {len(SPLITS)} splits")
    print(f"found {len(groups)} (class, ship) groups")
    if missing:
        print(f"WARNING: {len(missing)} rows had no matching .wav on disk "
              f"(first few): {missing[:5]}")

    OUT.mkdir(parents=True, exist_ok=True)


    per_class_ships = defaultdict(int)
    per_class_clips = defaultdict(int)
    grand_copied = 0

    for (cls, mmsi), items in sorted(groups.items()):
        items.sort(key=lambda x: x[0])
        dest_dir = OUT / f"{cls}_ship_ids" / f"ship_id_{mmsi}"
        dest_dir.mkdir(parents=True, exist_ok=True)
        for new_idx, (_, src) in enumerate(items):
            dest = dest_dir / f"{mmsi}_{new_idx}.wav"
            if MOVE:
                shutil.move(str(src), str(dest))
            else:
                shutil.copy2(str(src), str(dest))
            grand_copied += 1
        per_class_ships[cls] += 1
        per_class_clips[cls] += len(items)

    print("\n=== summary (per class) ===")
    print(f"{'class':16}{'ships':>7}{'clips':>9}")
    for cls in sorted(per_class_clips):
        print(f"{cls:16}{per_class_ships[cls]:>7}{per_class_clips[cls]:>9}")
    print("-" * 32)
    print(f"{'TOTAL':16}{sum(per_class_ships.values()):>7}{grand_copied:>9}")
    print(f"\n{'moved' if MOVE else 'copied'} {grand_copied} wav files -> {OUT}")


if __name__ == "__main__":
    main()

##Create Train 2000_4000

In [ ]:
#This script partitions ship ids within each class into a training dataset, ensuring no leakage into the validation or test sets.

import random
import shutil
from pathlib import Path


OUT = Path(".....Inputs/2000_4000_2000_4000_splits") #CHANGE THIS
TRAIN_DIR = Path("...OUTPUTS/2000_4000/train")#CHANGE THIS
SEED = 42

TARGETS = {
    "background": 688,
    "cargo": 688,
    "passengership": 688,
    "tanker": 688,
    "tug": 688,
}
BACKGROUND_CLASS = "background"


def list_ship_dirs(class_dir):
    if not class_dir.is_dir():
        return []
    ships = [d for d in sorted(class_dir.iterdir())
             if d.is_dir() and d.name.startswith("ship_id_") and not d.name.endswith("_used")]
    return ships


def wavs_in(ship_dir):
    return sorted(p for p in ship_dir.iterdir()
                  if p.suffix.lower() == ".wav" and not p.stem.endswith("_used"))


def mark_file_used(p):
    target = p.with_name(p.stem + "_used.wav")
    if not target.exists():
        p.rename(target)


def build_background(class_dir, dest_dir, target, rng):
    ships = list_ship_dirs(class_dir)
    if not ships:
        print(f"[background] no available ship folders in {class_dir} -> skip", flush=True)
        return 0

    all_wavs = []
    for s in ships:
        all_wavs.extend(wavs_in(s))
    if not all_wavs:
        print(f"[background] no unused wavs in {class_dir} -> skip", flush=True)
        return 0

    pool_size = len(all_wavs) // 3          # 1/3 for train, 1/3 val, 1/3 test
    if pool_size < 1:
        pool_size = 1
    pool = rng.sample(all_wavs, pool_size)
    rng.shuffle(pool)

    print(f"[background] {len(ships)} ship id(s), {len(all_wavs)} unused clips -> "
          f"file pool floor({len(all_wavs)}/3)={pool_size}, target {target}", flush=True)

    take = min(target, len(pool))
    for p in pool[:take]:
        shutil.copy2(str(p), str(dest_dir / p.name))


    for p in pool:
        mark_file_used(p)

    status = "OK" if take == target else f"SHORT by {target - take}"
    print(f"    copied {take}/{target} clips from the file pool "
          f"({pool_size} files marked _used) [{status}]", flush=True)
    if take < target:
        print(f"    !! the 1/3 file pool holds only {pool_size} clips (< {target})", flush=True)
    return take


def main():
    rng = random.Random(SEED)
    if not OUT.is_dir():
        print(f"source outputs dir not found: {OUT}. Creating automatically")
        OUT.mkdir(parents=True, exist_ok=True)

    TRAIN_DIR.mkdir(parents=True, exist_ok=True)
    print(f"SEED = {SEED}\n", flush=True)

    grand_total = 0
    for cls, target in TARGETS.items():
        class_dir = OUT / f"{cls}_ship_ids"
        dest_dir = TRAIN_DIR / cls
        dest_dir.mkdir(parents=True, exist_ok=True)


        if cls == BACKGROUND_CLASS:
            grand_total += build_background(class_dir, dest_dir, target, rng)
            continue

        ships = list_ship_dirs(class_dir)
        if not ships:
            print(f"[{cls}] no available ship folders in {class_dir} -> skip", flush=True)
            continue


        n_ships = len(ships) // 3 #partition by 3, 1/3 for train, 1/3 for validation, 1/3 for test
        if n_ships < 1:
            n_ships = 1
        n_ships = min(n_ships, len(ships))

        chosen = rng.sample(ships, n_ships)            # random ship selection using random seed
        quota = target // n_ships

        # pre-load and shuffle each chosen ship's clips
        pool = {}
        for s in chosen:
            w = wavs_in(s)
            rng.shuffle(w)
            pool[s] = w

        print(f"[{cls}] {len(ships)} ships available -> using {n_ships} "
              f"(floor quota {quota}/ship), target {target}", flush=True)

        copied = 0
        taken = {s: 0 for s in chosen}

        # pass 1: take the floor quota from each chosen ship
        for s in chosen:
            avail = pool[s]
            take = min(quota, len(avail), target - copied)
            for p in avail[taken[s]: taken[s] + take]:
                shutil.copy2(str(p), str(dest_dir / p.name))
            taken[s] += take
            copied += take
            if copied >= target:
                break

        #We take whatever is left in other ship ids until target is met
        if copied < target:
            for s in chosen:
                if copied >= target:
                    break
                avail = pool[s]
                remaining_in_ship = avail[taken[s]:]
                need = target - copied
                take = min(need, len(remaining_in_ship))
                for p in remaining_in_ship[:take]:
                    shutil.copy2(str(p), str(dest_dir / p.name))
                taken[s] += take
                copied += take

        # mark each consumed ship folder as used
        for s in chosen:
            used_name = s.parent / f"{s.name}_used"
            if not used_name.exists():
                s.rename(used_name)

        status = "OK" if copied == target else f"SHORT by {target - copied}"
        print(f"    copied {copied}/{target} clips from {n_ships} ships "
              f"[{status}]", flush=True)
        if copied < target:
            print(f"    !! not enough clips across the {n_ships} selected ships to "
                  f"reach {target}; consider allocating more ships to this class", flush=True)
        grand_total += copied

    print(f"\nTRAIN dataset written to {TRAIN_DIR}")
    print(f"total clips copied: {grand_total}")


if __name__ == "__main__":
    main()

Val for 2000_4000

In [ ]:
#This script partitions ship ids within each class into a validation dataset, drawing only
#from ships/files not already consumed by the train build, so there is no
#leakage between train, validation, and test.

import random
import shutil
from pathlib import Path


OUT = Path(".....Inputs/2000_4000_2000_4000_splits") #CHANGE THIS
VAL_DIR = Path("...OUTPUTS/2000_4000/validation")#CHANGE THIS
SEED = 42
DIVISOR = 2

TARGETS = {                        # required clip count per class in the validation set
    "background": 72,
    "cargo": 72,
    "passengership": 72,
    "tanker": 72,
    "tug": 72,
}
BACKGROUND_CLASS = "background"    # one ship id


def list_ship_dirs(class_dir):
    if not class_dir.is_dir():
        return []
    ships = [d for d in sorted(class_dir.iterdir())
             if d.is_dir() and d.name.startswith("ship_id_") and not d.name.endswith("_used")]
    return ships


def wavs_in(ship_dir):
    return sorted(p for p in ship_dir.iterdir()
                  if p.suffix.lower() == ".wav" and not p.stem.endswith("_used"))


def mark_file_used(p):
    target = p.with_name(p.stem + "_used.wav")
    if not target.exists():
        p.rename(target)


def build_background(class_dir, dest_dir, target, rng):
    ships = list_ship_dirs(class_dir)
    if not ships:
        print(f"[background] no available ship folders in {class_dir} -> skip", flush=True)
        return 0

    all_wavs = []
    for s in ships:
        all_wavs.extend(wavs_in(s))          # excludes train's _used files
    if not all_wavs:
        print(f"[background] no unused wavs left in {class_dir} -> skip", flush=True)
        return 0

    pool_size = len(all_wavs) // DIVISOR
    if pool_size < 1:
        pool_size = 1
    pool = rng.sample(all_wavs, pool_size)
    rng.shuffle(pool)

    print(f"[background] {len(ships)} ship id(s), {len(all_wavs)} unused clips -> "
          f"file pool floor({len(all_wavs)}/{DIVISOR})={pool_size}, target {target}", flush=True)

    take = min(target, len(pool))
    for p in pool[:take]:
        shutil.copy2(str(p), str(dest_dir / p.name))

    for p in pool:                            # mark the half used
        mark_file_used(p)

    status = "OK" if take == target else f"SHORT by {target - take}"
    print(f"    copied {take}/{target} clips from the file pool "
          f"({pool_size} files marked _used) [{status}]", flush=True)
    if take < target:
        print(f"    !! the 1/{DIVISOR} file pool holds only {pool_size} clips "
              f"(< {target})", flush=True)
    return take


def main():
    rng = random.Random(SEED)
    if not OUT.is_dir():
        raise SystemExit(f"source outputs dir not found: {OUT}")
    VAL_DIR.mkdir(parents=True, exist_ok=True)
    print(f"SEED = {SEED}\n", flush=True)

    grand_total = 0
    for cls, target in TARGETS.items():
        class_dir = OUT / f"{cls}_ship_ids"
        dest_dir = VAL_DIR / cls
        dest_dir.mkdir(parents=True, exist_ok=True)

        # background: single ship id
        if cls == BACKGROUND_CLASS:
            grand_total += build_background(class_dir, dest_dir, target, rng)
            continue

        ships = list_ship_dirs(class_dir)      # excludes train's _used ships
        if not ships:
            print(f"[{cls}] no available ship folders in {class_dir} -> skip", flush=True)
            continue

        # how many ships to use for validation
        n_ships = len(ships) // DIVISOR
        if n_ships < 1:
            n_ships = 1
        n_ships = min(n_ships, len(ships))

        chosen = rng.sample(ships, n_ships)            # random ship selection using random seed
        quota = target // n_ships

        # pre-load and shuffle each chosen ship's clips
        pool = {}
        for s in chosen:
            w = wavs_in(s)
            rng.shuffle(w)
            pool[s] = w

        print(f"[{cls}] {len(ships)} unused ships -> using {n_ships} "
              f"(floor quota {quota}/ship), target {target}", flush=True)

        copied = 0
        taken = {s: 0 for s in chosen}

        # pass 1: take the floor quota from each chosen ship
        for s in chosen:
            avail = pool[s]
            take = min(quota, len(avail), target - copied)
            for p in avail[taken[s]: taken[s] + take]:
                shutil.copy2(str(p), str(dest_dir / p.name))
            taken[s] += take
            copied += take
            if copied >= target:
                break

        # pass 2: fill the remaining shortfall from the other chosen ships,
        #We take whatever is left in other ship ids until target is met
        if copied < target:
            for s in chosen:
                if copied >= target:
                    break
                avail = pool[s]
                remaining_in_ship = avail[taken[s]:]
                need = target - copied
                take = min(need, len(remaining_in_ship))
                for p in remaining_in_ship[:take]:
                    shutil.copy2(str(p), str(dest_dir / p.name))
                taken[s] += take
                copied += take

        # mark each consumed ship folder as used
        for s in chosen:
            used_name = s.parent / f"{s.name}_used"
            if not used_name.exists():
                s.rename(used_name)

        status = "OK" if copied == target else f"SHORT by {target - copied}"
        print(f"    copied {copied}/{target} clips from {n_ships} ships "
              f"[{status}]", flush=True)
        if copied < target:
            print(f"    !! not enough clips across the {n_ships} selected ships to "
                  f"reach {target}; consider allocating more ships to this class", flush=True)
        grand_total += copied

    print(f"\nVALIDATION dataset written to {VAL_DIR}")
    print(f"total clips copied: {grand_total}")


if __name__ == "__main__":
    main()

##TEST for 2000_4000

In [ ]:
#This script partitions the remaining ship ids within each class into a test dataset. It draws
#only from ships/files not already consumed by the train and validation builds,
#completing a leakage-free three-way split.

import random
import shutil
from pathlib import Path


OUT = Path(".....Inputs/2000_4000_2000_4000_splits") #CHANGE THIS
TEST_DIR = Path("...OUTPUTS/2000_4000/test")#CHANGE THIS
SEED = 42
DIVISOR = 1                        # final split

TARGETS = {                        # required clip count per class in the test set
    "background": 40,
    "cargo": 40,
    "passengership": 40,
    "tanker": 40,
    "tug": 40,
}
BACKGROUND_CLASS = "background"    # one ship id


def list_ship_dirs(class_dir):
    """Unused ship_id_* folders in a class folder (skips ones already marked _used)."""
    if not class_dir.is_dir():
        return []
    ships = [d for d in sorted(class_dir.iterdir())
             if d.is_dir() and d.name.startswith("ship_id_") and not d.name.endswith("_used")]
    return ships


def wavs_in(ship_dir):
    """Unused wavs in a ship folder (skips files already marked _used)."""
    return sorted(p for p in ship_dir.iterdir()
                  if p.suffix.lower() == ".wav" and not p.stem.endswith("_used"))


def mark_file_used(p):
    """Rename a wav to <stem>_used.wav (records what this split consumed)."""
    target = p.with_name(p.stem + "_used.wav")
    if not target.exists():
        p.rename(target)


def build_background(class_dir, dest_dir, target, rng):
    """Background has a single ship id, so its FILES are partitioned instead of its ships.
    Train and validation already marked their thirds/halves _used; test uses ALL remaining
    files, draws the target from them, and marks them _used."""
    ships = list_ship_dirs(class_dir)
    if not ships:
        print(f"[background] no available ship folders in {class_dir} -> skip", flush=True)
        return 0

    all_wavs = []
    for s in ships:
        all_wavs.extend(wavs_in(s))
    if not all_wavs:
        print(f"[background] no unused wavs left in {class_dir} -> skip", flush=True)
        return 0

    pool_size = len(all_wavs) // DIVISOR
    if pool_size < 1:
        pool_size = 1
    pool = rng.sample(all_wavs, pool_size)
    rng.shuffle(pool)

    print(f"[background] {len(ships)} ship id(s), {len(all_wavs)} unused clips -> "
          f"file pool (all) {pool_size}, target {target}", flush=True)

    take = min(target, len(pool))
    for p in pool[:take]:
        shutil.copy2(str(p), str(dest_dir / p.name))

    for p in pool:
        mark_file_used(p)

    status = "OK" if take == target else f"SHORT by {target - take}"
    print(f"    copied {take}/{target} clips from the file pool "
          f"({pool_size} files marked _used) [{status}]", flush=True)
    if take < target:
        print(f"    !! only {pool_size} clips remained (< {target}); this was the last split",
              flush=True)
    return take


def main():
    rng = random.Random(SEED)
    if not OUT.is_dir():
        raise SystemExit(f"source outputs dir not found: {OUT}")
    TEST_DIR.mkdir(parents=True, exist_ok=True)
    print(f"SEED = {SEED}\n", flush=True)

    grand_total = 0
    for cls, target in TARGETS.items():
        class_dir = OUT / f"{cls}_ship_ids"
        dest_dir = TEST_DIR / cls
        dest_dir.mkdir(parents=True, exist_ok=True)

        # background: single ship id
        if cls == BACKGROUND_CLASS:
            grand_total += build_background(class_dir, dest_dir, target, rng)
            continue

        ships = list_ship_dirs(class_dir)
        if not ships:
            print(f"[{cls}] no available ship folders in {class_dir} -> skip", flush=True)
            continue

        # how many ships to use for test
        n_ships = len(ships) // DIVISOR
        if n_ships < 1:
            n_ships = 1
        n_ships = min(n_ships, len(ships))

        chosen = rng.sample(ships, n_ships)            # random ship selection using random seed
        quota = target // n_ships

        # pre-load and shuffle each chosen ship's clips
        pool = {}
        for s in chosen:
            w = wavs_in(s)
            rng.shuffle(w)
            pool[s] = w

        print(f"[{cls}] {len(ships)} unused ships -> using {n_ships} "
              f"(floor quota {quota}/ship), target {target}", flush=True)

        copied = 0
        taken = {s: 0 for s in chosen}

        # pass 1: take the floor quota from each chosen ship
        for s in chosen:
            avail = pool[s]
            take = min(quota, len(avail), target - copied)
            for p in avail[taken[s]: taken[s] + take]:
                shutil.copy2(str(p), str(dest_dir / p.name))
            taken[s] += take
            copied += take
            if copied >= target:
                break

        # pass 2: fill the remaining shortfall from the other chosen ships,
        #We take whatever is left in other ship ids until target is met
        if copied < target:
            for s in chosen:
                if copied >= target:
                    break
                avail = pool[s]
                remaining_in_ship = avail[taken[s]:]
                need = target - copied
                take = min(need, len(remaining_in_ship))
                for p in remaining_in_ship[:take]:
                    shutil.copy2(str(p), str(dest_dir / p.name))
                taken[s] += take
                copied += take

        # mark each consumed ship folder as used
        for s in chosen:
            used_name = s.parent / f"{s.name}_used"
            if not used_name.exists():
                s.rename(used_name)

        status = "OK" if copied == target else f"SHORT by {target - copied}"
        print(f"    copied {copied}/{target} clips from {n_ships} ships "
              f"[{status}]", flush=True)
        if copied < target:
            print(f"    !! not enough clips across the {n_ships} remaining ships to "
                  f"reach {target}; this was the last split", flush=True)
        grand_total += copied

    print(f"\nTEST dataset written to {TEST_DIR}")
    print(f"total clips copied: {grand_total}")


if __name__ == "__main__":
    main()

##REPEAT THE STEPS FOR 3000_5000 AND 4000_6000, JUST MAKE SURE TO CHANGE THE FILE PATHS!

In [ ]:
#This script groups the classified clips into class/MMSI folders and writes a
#manifest + summary CSV describing what was placed where.

import os
import shutil
import wave

import pandas as pd

from tqdm import tqdm


# ======================= EDIT THESE =======================
METADATA_ROOT = "....../07b_classified_wav_files/inclusion_2000_exclusion_4000"  #CHANGE THIS: folder holding the metadata CSV; clip paths in the CSV are relative to it
METADATA_FILE = "metadata_1s.csv"                                                #CHANGE THIS: metadata CSV file name inside METADATA_ROOT
OUTPUT_DIR = "Outputs/ONC/ship_splits"        #CHANGE THIS: where the class/MMSI folders will be written

UNIT = "file"       # "file" = place whole WAV files, "segment" = cut into fixed length segments
MODE = "hardlink"   # "hardlink", "symlink" or "copy" — only used when UNIT is "file"
SECONDS = 1         # duration of each segment — only used when UNIT is "segment"
# ==========================================================


BACKGROUND_LABEL = "background"
# The background class has no vessel identity, so every background clip is
# stored under a single MMSI folder.
BACKGROUND_MMSI = 0


class bcolors:
    HEADER = "\033[95m"
    WARNING = "\033[93m"
    ENDC = "\033[0m"


def create_dir(parent, name):
    directory = os.path.join(parent, name)
    os.makedirs(directory, exist_ok=True)
    return directory


def get_mmsi_folder_name(mmsi):
    return str(int(mmsi))


def get_clips_from_metadata(metadata_file):
    """Categorise every clip in the metadata by its class and MMSI.

    Returns one row per WAV file with the label, the MMSI folder name it
    belongs to and the 1 second offsets that the metadata lists for it.
    """
    metadata = pd.read_csv(metadata_file)

    clips = []
    for path, rows in metadata.groupby("path"):
        labels = rows.label.unique()
        mmsis = rows.MMSI.unique()

        if len(labels) > 1:
            print(f"{bcolors.WARNING}Skipping {path}: more than one label {labels}{bcolors.ENDC}")
            continue
        if len(mmsis) > 1:
            print(f"{bcolors.WARNING}Skipping {path}: more than one MMSI {mmsis}{bcolors.ENDC}")
            continue

        label = labels[0]
        mmsi = BACKGROUND_MMSI if label == BACKGROUND_LABEL else mmsis[0]

        clips.append(
            {
                "label": label,
                "mmsi": get_mmsi_folder_name(mmsi),
                "path": path,
                "sub_inits": sorted(rows.sub_init.tolist()),
            }
        )

    return clips


def place_wav_file(source_file, destination_file, mode):
    if os.path.exists(destination_file):
        os.remove(destination_file)

    if mode == "hardlink":
        os.link(source_file, destination_file)
    elif mode == "symlink":
        os.symlink(os.path.abspath(source_file), destination_file)
    else:
        shutil.copyfile(source_file, destination_file)


def save_wav_segments(source_file, destination_directory, sub_inits, seconds):
    """Cut the clip into the fixed length segments listed in the metadata."""
    saved = []
    file_name = os.path.splitext(os.path.basename(source_file))[0]

    with wave.open(source_file, "rb") as source_wav:
        frame_rate = source_wav.getframerate()
        segment_frames = frame_rate * seconds
        total_frames = source_wav.getnframes()

        for sub_init in sub_inits:
            start_frame = sub_init * frame_rate
            if start_frame + segment_frames > total_frames:
                print(
                    f"{bcolors.WARNING}Skipping {source_file} at {sub_init}s: "
                    f"beyond the end of the file{bcolors.ENDC}"
                )
                continue

            source_wav.setpos(start_frame)
            frames = source_wav.readframes(segment_frames)

            destination_file = os.path.join(
                destination_directory, f"{file_name}_{sub_init:05d}.wav"
            )
            with wave.open(destination_file, "wb") as destination_wav:
                destination_wav.setnchannels(source_wav.getnchannels())
                destination_wav.setsampwidth(source_wav.getsampwidth())
                destination_wav.setframerate(frame_rate)
                destination_wav.writeframes(frames)

            saved.append((destination_file, sub_init))

    return saved


def group_clips_by_mmsi(metadata_root, metadata_file, output_directory, unit, mode, seconds):
    clips = get_clips_from_metadata(os.path.join(metadata_root, metadata_file))
    print(f"Categorised {len(clips)} clips by class and MMSI")

    manifest = []
    for clip in tqdm(clips, total=len(clips)):
        # The metadata stores the clip paths relative to the metadata folder.
        source_file = os.path.join(metadata_root, clip["path"])
        if not os.path.exists(source_file):
            print(f"{bcolors.WARNING}Skipping {clip['path']}: file not found{bcolors.ENDC}")
            continue

        # Every MMSI folder lives inside the folder of its own class.
        label_directory = create_dir(output_directory, clip["label"])
        mmsi_directory = create_dir(label_directory, clip["mmsi"])

        if unit == "segment":
            saved = save_wav_segments(source_file, mmsi_directory, clip["sub_inits"], seconds)
        else:
            destination_file = os.path.join(mmsi_directory, os.path.basename(source_file))
            place_wav_file(source_file, destination_file, mode)
            saved = [(destination_file, None)]

        for destination_file, sub_init in saved:
            manifest.append(
                {
                    "label": clip["label"],
                    "mmsi": clip["mmsi"],
                    "group_id": f"{clip['label']}/{clip['mmsi']}",
                    "source_path": clip["path"],
                    "sub_init": sub_init,
                    "path": os.path.relpath(destination_file, output_directory),
                }
            )

    return pd.DataFrame(manifest)


def save_manifest(manifest, output_directory):
    manifest_file = os.path.join(output_directory, "grouped_manifest.csv")
    manifest.to_csv(manifest_file, index=False)

    summary = (
        manifest.groupby("label")
        .agg(mmsi_folders=("mmsi", "nunique"), clips=("path", "count"))
        .sort_values("clips", ascending=False)
    )
    summary_file = os.path.join(output_directory, "grouped_summary.csv")
    summary.to_csv(summary_file)

    print(f"\n{summary.to_string()}")
    print(f"\nTotal MMSI folders: {manifest.group_id.nunique()}")
    print(f"Manifest saved to {manifest_file}")
    print(f"Summary saved to {summary_file}")


def main():
    if not os.path.isdir(METADATA_ROOT):
        raise SystemExit(f"metadata root not found: {METADATA_ROOT}")
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    print(f"\n{bcolors.HEADER}Grouping clips by class and MMSI{bcolors.ENDC}")
    print(f"Reading {os.path.join(METADATA_ROOT, METADATA_FILE)}")
    print(f"Writing to {OUTPUT_DIR}")

    manifest = group_clips_by_mmsi(
        METADATA_ROOT,
        METADATA_FILE,
        OUTPUT_DIR,
        UNIT,
        MODE,
        SECONDS,
    )
    save_manifest(manifest, OUTPUT_DIR)


if __name__ == "__main__":
    main()
